In [1]:
import numpy as np

def update_cg(var, r, q, Aq, rold):
    alpha = rold / np.inner(q, Aq)
    var = var + alpha * q
    r = r - alpha * Aq
    rnew = np.inner(r, r)
    q = r + (rnew / rold) * q
    return var, r, q, rnew

def compute_mae(var, var_hat):
    return np.sum(np.abs(var - var_hat)) / var.shape[0]

def compute_rmse(var, var_hat):
    return np.sqrt(np.sum((var - var_hat) ** 2) / var.shape[0])

def forecast_multi_step(series, mu, W, V, d, horizon, step = 1):
    N, t = series.shape
    origins = np.arange(t - horizon, t) - step + 1                  # (horizon,)
    idx = origins[:, None] - (np.arange(d) + 1)[None, :]            # (horizon, d)
    L = series[:, idx].transpose(1, 2, 0)                           # (horizon, d, N)
    for _ in range(step):
        Xt = L.reshape(horizon, d * N)
        pred = mu + (Xt @ V.T) @ W.T                                # (horizon, N)
        L = np.concatenate([pred[:, None, :], L[:, :-1, :]], axis=1)
    return pred.T

In [2]:
def solve_mu(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    if N * d * (T - d) <= 1e+8:
        X = np.empty((N * d, T - d))
        for k in range(d):
            X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
        C = Y[:, d :] - W @ V @ X
    else:
        temp = np.empty((R, T - d))
        for t in range(d, T):
            temp[:, t - d] = V @ np.hstack([Y[:, t - k - 1] for k in range(d)])
        C = Y[:, d :] - W @ temp
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = Y[:, d :] - mu[:, np.newaxis] - C
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        mu = np.einsum('nt, nt -> n', theta, C, optimize = 'greedy') / np.sum(theta, axis = 1)
    return mu

In [3]:
def ell_W(temp, W):
    return np.einsum('nwk, nw -> nk', temp, W, optimize = 'greedy').reshape(-1, order = 'F')

def solve_W(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    if N * d * (T - d) <= 1e+8:
        X = np.empty((N * d, T - d))
        for k in range(d):
            X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
        var1 = V @ X
    else:
        var1 = np.empty((R, T - d))
        for t in range(d, T):
            var1[:, t - d] = V @ np.hstack([Y[:, t - k - 1] for k in range(d)])
    temp1 = np.einsum('wt, kt -> wkt', var1, var1, optimize = 'greedy')
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - W @ var1
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        w = np.reshape(W, -1, order = 'F')
        response = np.einsum('nt, wt -> nw', theta * data, var1, optimize = 'greedy').reshape(-1, order = 'F')
        temp = np.einsum('nt, wkt -> nwk', theta, temp1, optimize = 'greedy')
        r = response - ell_W(temp, W)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (N, R), order = 'F')
            Aq = ell_W(temp, Q)
            w, r, q, rold = update_cg(w, r, q, Aq, rold)
        W = np.reshape(w, (N, R), order = 'F')
    return W

In [4]:
def ell_V(temp, X, V):
    return np.einsum('kht, ht, lt -> kl', temp, V @ X, X, optimize = 'greedy').reshape(-1, order = 'F')

def solve_V(Y, p, mu, W, V, delta, d, R, max_iter = 50):
    N, T = Y.shape
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    data = Y[:, d :] - mu[:, np.newaxis]
    for iter in range(max_iter):
    # for iter in range(1):
        epsilon = data - W @ (V @ X)
        theta = np.ones((N, T - d))
        pos = np.where(np.abs(epsilon) > delta)
        theta[pos] = delta / np.abs(epsilon[pos])
        ######### p = 0 is the log-Huber ##########
        theta[pos] = delta ** (2 - p) / (np.abs(epsilon[pos]) ** (2 - p))
        ###########################################

        v = np.reshape(V, -1, order = 'F')
        response = (W.T @ (theta * data) @ X.T).reshape(-1, order = 'F')
        temp = np.einsum('nt, nk, nh -> kht', theta, W, W, optimize = 'greedy')
        r = response - ell_V(temp, X, V)
        q = r.copy()
        rold = np.inner(r, r)
        for it in range(5): # CG iteration
            Q = np.reshape(q, (R, d * N), order = 'F')
            Aq = ell_V(temp, X, Q)
            v, r, q, rold = update_cg(v, r, q, Aq, rold)
        V = np.reshape(v, (R, d * N), order = 'F')
    return V

In [5]:
import numpy as np
import time

def ell_v_initial(var, X, V):
    return (var @ (V @ X) @ X.T).reshape(-1, order = 'F')

def initialize_WV(Y, d, R):
    N, T = Y.shape
    X = np.zeros((N * d, T - d))
    for k in range(d):
        X[k * N : (k + 1) * N, :] = Y[:, d - k - 1 : T - k - 1]
    mat, _, _ = np.linalg.svd(Y[:, d :], full_matrices = False)
    W = mat[:, : R]
    V = np.zeros((R, d * N))
    v = np.reshape(V, -1, order = 'F')
    response = np.einsum('nr, nt, kt -> rk', W, Y[:, d :], X, optimize = 'greedy').reshape(-1, order = 'F')
    var = W.T @ W
    r = response - ell_v_initial(var, X, V)
    q = r.copy()
    rold = np.inner(r, r)
    for it in range(5): # CG iteration
        Q = np.reshape(q, (R, d * N), order = 'F')
        Aq = ell_v_initial(var, X, Q)
        v, r, q, rold = update_cg(v, r, q, Aq, rold)
    V = np.reshape(v, (R, d * N), order = 'F')
    return W, V

def huber_var_mf(Y, p, delta, d, R, max_iter = 50):
    N = Y.shape[0]
    mu = np.zeros(N)
    W, V = initialize_WV(Y, d, R)
    for iter in range(max_iter):
        mu = solve_mu(Y, p, mu, W, V, delta, d, R)
        W = solve_W(Y, p, mu, W, V, delta, d, R)
        V = solve_V(Y, p, mu, W, V, delta, d, R)
    return mu, W, V

In [6]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

step = 12
for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [3]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : 17 * 24]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast_multi_step(np.array(data[:, : 24 * 24]), mu, W, V, d, horizon, step)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast_multi_step(np.array(data), mu, W, V, d, horizon, step)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 3
Delta = 40
Running time: 504 seconds
MAE: 213.004
RMSE: 1178.86

MAE: 289.602
RMSE: 1552.59

p = 0.6666666666666666
Rank = 3
Delta = 40
Running time: 499 seconds
MAE: 220.72
RMSE: 1206.12

MAE: 302.844
RMSE: 1599.79

p = 0.5
Rank = 3
Delta = 40
Running time: 499 seconds
MAE: 221.526
RMSE: 1346.12

MAE: 296.794
RMSE: 1655.42

p = 0.3333333333333333
Rank = 3
Delta = 40
Running time: 499 seconds
MAE: 230.812
RMSE: 1390.58

MAE: 314.587
RMSE: 1739.34

p = 0
Rank = 3
Delta = 40
Running time: 499 seconds
MAE: 228.099
RMSE: 1317.54

MAE: 306.907
RMSE: 1671.66



In [7]:
import time
import pandas as pd
import cupy as np

data = pd.read_csv('top_pages.csv.gz', compression='gzip').iloc[:, 2 :].values

step = 12
for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 168
    d = 7 * 24
    for R in [4]:
        for delta in [40]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : 17 * 24]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, 24 * 17 : 24 * 24])
            est = forecast_multi_step(np.array(data[:, : 24 * 24]), mu, W, V, d, horizon, step)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast_multi_step(np.array(data), mu, W, V, d, horizon, step)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 4
Delta = 40
Running time: 503 seconds
MAE: 204.641
RMSE: 1145.55

MAE: 282.743
RMSE: 1530.64

p = 0.6666666666666666
Rank = 4
Delta = 40
Running time: 503 seconds
MAE: 207.871
RMSE: 1141.86

MAE: 289.304
RMSE: 1552.58

p = 0.5
Rank = 4
Delta = 40
Running time: 503 seconds
MAE: 211.827
RMSE: 1179.42

MAE: 295.836
RMSE: 1619.54

p = 0.3333333333333333
Rank = 4
Delta = 40
Running time: 503 seconds
MAE: 209.589
RMSE: 1313.43

MAE: 283.524
RMSE: 1600.54

p = 0
Rank = 4
Delta = 40
Running time: 503 seconds
MAE: 209.111
RMSE: 1161.07

MAE: 287.086
RMSE: 1552.34



In [7]:
import numpy as np

data = np.load('daymet_tmax_na_2010.npz')['arr_0']
for t in [11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]:
    data = np.append(data, np.load('daymet_tmax_na_20{}.npz'.format(t))['arr_0'], axis = 1)

In [13]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [5]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : - (365 + 365)]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (365 + 365) : - 365])
            est = forecast_multi_step(np.array(data[:, : - 365]), mu, W, V, d, horizon)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast_multi_step(np.array(data), mu, W, V, d, horizon)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 5
Delta = 1
Running time: 761 seconds
MAE: 2.80841
RMSE: 3.74799

MAE: 2.82798
RMSE: 3.72548

p = 0.6666666666666666
Rank = 5
Delta = 1
Running time: 761 seconds
MAE: 2.82135
RMSE: 3.78044

MAE: 2.83986
RMSE: 3.75197

p = 0.5
Rank = 5
Delta = 1
Running time: 761 seconds
MAE: 2.82761
RMSE: 3.79513

MAE: 2.84584
RMSE: 3.76429

p = 0.3333333333333333
Rank = 5
Delta = 1
Running time: 761 seconds
MAE: 2.83848
RMSE: 3.81679

MAE: 2.8552
RMSE: 3.78138

p = 0
Rank = 5
Delta = 1
Running time: 763 seconds
MAE: 2.85907
RMSE: 3.85804

MAE: 2.86958
RMSE: 3.81053



In [14]:
import time
import cupy as np

for p in [1, 2/3, 1/2, 1/3, 0]:
    print('p = {}'.format(p))
    horizon = 365
    d = 30
    for R in [6]:
        for delta in [1]:
            print('Rank = {}'.format(R))
            print('Delta = {}'.format(delta))
            start = time.time()
            np.random.seed(0)
            mu, W, V = huber_var_mf(np.array(data[:, : - (365 + 365)]), p, delta, d, R)
            end = time.time()
            print('Running time: %d seconds'%(end - start))

            ground = np.array(data[:, - (365 + 365) : - 365])
            est = forecast_multi_step(np.array(data[:, : - 365]), mu, W, V, d, horizon)
            print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
            print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
            print()

    ground = np.array(data[:, - horizon :])
    est = forecast_multi_step(np.array(data), mu, W, V, d, horizon)
    print('MAE: {:.6}'.format(compute_mae(ground.reshape(-1), est.reshape(-1))))
    print('RMSE: {:.6}'.format(compute_rmse(ground.reshape(-1), est.reshape(-1))))
    print()

p = 1
Rank = 6
Delta = 1
Running time: 761 seconds
MAE: 2.72519
RMSE: 3.61033

MAE: 2.75244
RMSE: 3.60308

p = 0.6666666666666666
Rank = 6
Delta = 1
Running time: 763 seconds
MAE: 2.74315
RMSE: 3.65821

MAE: 2.76645
RMSE: 3.63836

p = 0.5
Rank = 6
Delta = 1
Running time: 762 seconds
MAE: 2.75161
RMSE: 3.68677

MAE: 2.77318
RMSE: 3.66034

p = 0.3333333333333333
Rank = 6
Delta = 1
Running time: 762 seconds
MAE: 2.76255
RMSE: 3.71765

MAE: 2.7808
RMSE: 3.68247

p = 0
Rank = 6
Delta = 1
Running time: 760 seconds
MAE: 2.78574
RMSE: 3.77157

MAE: 2.79843
RMSE: 3.72277

